In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import (
    OneHotEncoder, 
    LabelEncoder, 
    OrdinalEncoder, 
    PowerTransformer, 
    StandardScaler,
    MinMaxScaler
)

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

## Load Data

In [2]:
us_visa_data = pd.read_csv("EasyVisa.csv")
us_visa_data.head()

,case_id,continent,education_of_employee,has_job_experience,requires_job_training,no_of_employees,yr_of_estab,region_of_employment,prevailing_wage,unit_of_wage,full_time_position,case_status
0,EZYV01,Asia,High School,N,N,14513,2007,West,592.2029,Hour,Y,Denied
1,EZYV02,Asia,Master's,Y,N,2412,2002,Northeast,83425.6500,Year,Y,Certified
2,EZYV03,Asia,Bachelor's,N,Y,44444,2008,West,122996.8600,Year,Y,Denied
3,EZYV04,Asia,Bachelor's,N,N,98,1897,West,83434.0300,Year,Y,Denied
4,EZYV05,Africa,Master's,Y,N,1082,2005,South,149907.3900,Year,Y,Certified


## Data Cleaning

During initial analysis we found that that the `case_id` has only unique values therefore high variance which is normal given that it's the unique identifier of each visa application however this feature is not useful for modeling so it can be safely removed. We also saw that sone of the records have negative values for the `no_of_employees` feature. It seems that this is a mistake during data collection since there is no pattern in the data that can suggest other reason. These records are very few so they can be safely removed too. 

In [3]:
us_visa_data = us_visa_data.drop(columns=["case_id"])

negative_no_employees = us_visa_data[us_visa_data["no_of_employees"] < 0 ].index
us_visa_data = us_visa_data.drop(negative_no_employees)

us_visa_data.shape

(25447, 11)

## Feature Engineering

In [4]:
from datetime import date

today_date = date.today()
current_year = today_date.year

# Creating a campany age feature to facilitate any future models
us_visa_data["company_age"] = current_year - us_visa_data["yr_of_estab"] 
us_visa_data = us_visa_data.drop(columns=["yr_of_estab"])

us_visa_data.shape

(25447, 11)

## Preprocessing

Before training any machine learning model the data has to be in a suitable format for this task. Machine learning models work only with numbers and don't know what to do with categories so categorical features have to be encoded in some way. Also some models don't work well with numerical features that are on vastly different scales and those features have to be rescaled to some range.

In [5]:
numeric_features = [feature for feature in us_visa_data.columns if us_visa_data[feature].dtype != 'str']
ohe_features = [
    "continent", 
    "has_job_experience", 
    "requres_job_training", 
    "region_of_employment", 
    "unit_of_wage", 
    "full_time_position"
]

# The categories inside education_of_employee have an inherent order 
# (e.g. High School education is lower than Bachelor's degree etc.)
ordinal_features = ["education_of_employee"]
target_column = ["case_status"]
education_categories = ["High School", "Bachelor's", "Master's", "Doctorate"]

numeric_pipeline = Pipeline(steps=[
    # Makes skewed distributions more Gaussian-like. May algorithms assume normally distributed data
    ("log_transform", PowerTransformer(standardize=False)),
    
    # Scales the data to be in the range between 0 and 1
    # Suitable for normally distributed data with no outliers 
    ("scale", MinMaxScaler())
])

ohe_pipeline = Pipeline([
    # Creates a new binary column for each category
    ("ohe", OneHotEncoder(sparse_output=False, drop="first")),
])

ordinal_pipeline = Pipeline([
    ("encode", OrdinalEncoder(categories=[education_categories]))
]) 

transformer = ColumnTransformer([
    ("numeric_pipeline", numeric_pipeline, numeric_features),
    ("ohe_pipeline", ohe_pipeline, ohe_features),
    ("ordinal_pipeline", ordinal_pipeline, ordinal_features)
])

# Encoding the target column as binary labels (0 -> Denied, 1 -> Certified)
us_visa_data["case_status"] = us_visa_data["case_status"].apply(lambda row: 0 if row == "Denied" else 1)

In [6]:
us_visa_data["case_status"]

0        0
1        1
2        0
3        0
4        1
        ..
25475    1
25476    1
25477    1
25478    1
25479    1
Name: case_status, Length: 25447, dtype: int64

## Model Building

### Train, Validation and Test Splits

In [ ]:
visa_attributes = us_visa_data.drop(columns=target_column) # Input features
visa_target = us_visa_data[target_column] # Target column



